# Day 3 — A complete RAG loop over the AI Media Dataset

*anatoolbox try-out notebook · HSLU Computational Language Technologies capstone*

Day 2 retrieved passages. Today the loop closes: questions are rewritten, articles are
chunked, candidates are reranked, and a language model writes an answer that cites its
sources — with every step recorded.

| New today | What it gives you in Stage 3 |
|---|---|
| **Any OpenAI-compatible LLM** | Use OpenAI, a hosted provider, or a model on your own machine — configuration, not code |
| `chunk_by_size` | The chunking baseline — fixed-size token windows, optional overlap, title context — for your own strategies to beat |
| `rewrite_query_for_retrieval` | Query expansion and decomposition — a listed Stage 3 enhancement |
| `retrieve_passages(queries_input=…)` | Retrieve with every rewrite at once, fused, with lineage |
| `rerank_passages` | Re-score candidates on their full text — another listed enhancement |
| recency and metadata filters | Time-aware retrieval for trend questions |
| `synthesize_answer` | Grounded answers with numbered citations that are *checked* |
| **Pipeline mode with provenance** | Results pass straight from tool to tool, and each records how it was made |

**What stays yours:** the chunk size, the rewriting strategy, the models, the answer
instructions — and, from Day 4, how you evaluate all of it. The tools make those choices
cheap to vary; they do not make them for you.

⏱ About 5 minutes on a CPU with a small local model. Sections that need a language model
are skipped cleanly if none is configured; reranking is skipped without `sentence-transformers`.

## 0 · Setup

In [1]:
import importlib.util
import os
import platform
import time

import pandas as pd
from IPython.display import Markdown, display

import anatoolbox

pd.set_option("display.max_colwidth", 110)
HAS_EMBEDDINGS = importlib.util.find_spec("sentence_transformers") is not None
print("python     :", platform.python_version())
print("anatoolbox :", anatoolbox.__version__)
print("reranking with a cross-encoder:", "available" if HAS_EMBEDDINGS else "not installed — the rerank section will be skipped")

python     : 3.11.8
anatoolbox : 0.1.0
reranking with a cross-encoder: available


## 1 · Choose a language model

Tools that need a language model talk to any **OpenAI-compatible endpoint**. Pick one line
below, or set `ANATOOLBOX_LLM_BASE_URL`, `ANATOOLBOX_LLM_API_KEY` and `ANATOOLBOX_LLM_MODEL`
before starting Jupyter. There is no default model on purpose: which model wrote an answer
is part of the answer.

Two course-specific notes:

- **Roles.** Query rewriting asks for the `fast` role, answer writing for the `strong` role.
  Map them to different models with `configure_llm(models={"fast": ..., "strong": ...})`.
- **Stage 3 requires a different LLM to generate your Q&A pairs than the one inside your RAG
  pipeline.** Keep the two configurations clearly apart.

In [2]:
from anatoolbox.llm_client import call_llm_text, configure_llm, llm_settings

# Pick ONE (or configure through environment variables):
# configure_llm(model="gpt-4o-mini")                                                  # OpenAI — needs OPENAI_API_KEY
# configure_llm(base_url="http://localhost:11434/v1", model="qwen3:4b")                 # Ollama on your machine
# configure_llm(base_url="https://openrouter.ai/api/v1", api_key="...", model="qwen/qwen3-8b")

settings = llm_settings()
try:
    started = time.perf_counter()
    call_llm_text("Reply with one word.", "Say: ready", max_tokens=5)
    LLM_READY = True
    status = f"responding ({time.perf_counter() - started:.1f} s)"
except Exception as exc:  # no model configured, unreachable endpoint, bad key, ...
    LLM_READY = False
    status = f"not available — {type(exc).__name__}: {str(exc)[:160]}"

print("endpoint :", settings.base_url or "OpenAI")
print("models   :", settings.models or "none configured")
print("status   :", status)

endpoint : http://127.0.0.1:8765/v1
models   : {'default': 'Qwen3-0.6B'}
status   : responding (0.9 s)


## 2 · Load the dataset and pick your track

Same as Day 2: the dataset downloads from Kaggle's public API (no account needed) and is
cached in `data/`; set `AI_MEDIA_CSV` to use an existing copy. This notebook works on the
articles of **one topic track**, which is how Stage 3 is scoped.

In [3]:
import io
import urllib.request
import zipfile
from pathlib import Path

DATA_DIR = Path(os.environ.get("AI_MEDIA_DATA_DIR", "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/jannalipenkova/ai-media-dataset"


def find_or_download_csv():
    explicit = os.environ.get("AI_MEDIA_CSV")
    if explicit and Path(explicit).exists():
        return Path(explicit)
    cached = sorted(DATA_DIR.glob("ai_media_dataset_*.csv"))
    if cached:
        return cached[-1]
    print("Downloading the AI Media Dataset from Kaggle (~58 MB) …")
    with urllib.request.urlopen(KAGGLE_URL, timeout=180) as response:
        payload = response.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        name = next(n for n in archive.namelist() if n.endswith(".csv"))
        archive.extract(name, DATA_DIR)
    return DATA_DIR / name


TRACKS = {
    "Hardware & Infrastructure": r"\bgpus?\b|accelerator|nvidia|data cent(?:er|re)|semiconductor|\btsmc\b|\bchips?\b",
    "Foundation Models": r"foundation model|large language model|\bllms?\b|gpt-?\d|\bllama\b|gemini|claude|mistral|\bqwen\b",
    "Agentic Web": r"ai agents?|agentic|\bmcp\b|model context protocol|agent2agent|\ba2a\b|browser agent|computer use",
}
QUESTIONS = {
    "Agentic Web": "What security risks do autonomous browser agents create, and how are companies responding?",
    "Hardware & Infrastructure": "How are export controls affecting AI chip supply, and how are chipmakers adapting?",
    "Foundation Models": "Which open-weight foundation models were released in 2025, and how do they compare with closed models?",
}
TRACK = "Agentic Web"   # ← "Hardware & Infrastructure" | "Foundation Models" | "Agentic Web"
QUESTION = QUESTIONS[TRACK]

df = pd.read_csv(find_or_download_csv())
haystack = (df["title"] + " " + df["content"] + " " + df["tags"]).str.lower()
track_df = df[haystack.str.contains(TRACKS[TRACK], regex=True)]
track_csv = DATA_DIR / "day3_track_articles.csv"
track_df.to_csv(track_csv, index=False)
print(f"{TRACK}: {len(track_df):,} of {len(df):,} articles, {track_df['date'].min()} → {track_df['date'].max()}")
print("question:", QUESTION)

Agentic Web: 2,572 of 16,527 articles, 2024-09-11 → 2025-08-24
question: What security risks do autonomous browser agents create, and how are companies responding?


**Pipeline mode.** A notebook needs no memory layer. Each tool's result is passed straight into
the next tool as `input`, and every result carries a `provenance` block: a `run_id`, the tool,
its effective settings, and the `run_id`s it was derived from. (Agents attach recordset memory
instead, so a language model can refer to results by short handles rather than copying data.)

In [4]:
from anatoolbox import ToolContext, resolve_tools

anatoolbox.register_reference_tools()
ctx = ToolContext()  # pipeline mode: no memory — results are passed on as the next tool's input
ingest, chunk, rewrite, retrieve, rerank, synthesize = resolve_tools(
    ["ingest_corpus", "chunk_by_size", "rewrite_query_for_retrieval",
     "retrieve_passages", "rerank_passages", "synthesize_answer"]
)
print(anatoolbox.tools_by_stage())

articles = ingest.run({"path": str(track_csv), "name": "track_articles", "text_field": "content", "id_field": "Unnamed: 0"}, context=ctx)
print(f"ingested {articles['records']:,} articles · run {articles['provenance']['run_id']}")

{'preprocess': ['chunk_by_size'], 'gather': ['ingest_corpus', 'rerank_passages', 'retrieve_passages', 'rewrite_query_for_retrieval'], 'enrich': ['synthesize_answer']}


ingested 2,572 articles · run ingest_corpus-b4fe4cee


## 3 · Chunking: the baseline

Embedding models and rerankers read a few hundred tokens; a whole article is far longer
(Day 2 measured a median of ~1,550 tokens). `chunk_by_size` is the **baseline**: it cuts every
article into windows of `size` tokens, optionally overlapping by `overlap` tokens, with no
regard for paragraphs, sentences or meaning.

That is on purpose. Structural chunking (paragraphs, sentences), semantic chunking (split where
the topic shifts) and contextual chunking are **Stage 3 optimizations for you to build** and
measure against this baseline.

**Chunk size is a trade-off, not a constant.** Small chunks match specific facts precisely but
lose context; large chunks keep context but dilute the match and crowd the prompt. The table
shows what `size` does to this corpus.

In [5]:
from anatoolbox.corpus import get_corpus
from anatoolbox.preprocess.chunk.chunk_by_size import chunk_text_by_size

source = get_corpus("track_articles")
rows = []
for size in (100, 200, 400):
    lengths = [c["tokens"] for text in source.texts() for c in chunk_text_by_size(text, size=size)]
    rows.append({"size (words)": size, "chunks": len(lengths), "median length": int(pd.Series(lengths).median()),
                 "shorter than size": f"{sum(n < size for n in lengths) / len(lengths):.0%}"})
display(pd.DataFrame(rows))

,size (words),chunks,median length,shorter than size
0,100,41345,100,6%
1,200,21322,200,12%
2,400,11352,400,23%


In [6]:
chunks = chunk.run({"input": articles, "size": 200, "overlap": 40}, context=ctx)
print(f"{chunks['chunks']:,} chunks from {chunks['articles']:,} articles · run {chunks['provenance']['run_id']}, derived from {chunks['provenance']['derived_from']}")
print("tokens per chunk:", chunks["tokens"])

chunk_corpus = get_corpus(chunks["corpus"])
first_article = chunk_corpus.records[0]["source_id"]
display(pd.DataFrame([
    {"id": r["id"], "tokens": f"{r['token_start']}–{r['token_end']}",
     "starts with": r["chunk_text"][:60].replace("\n", " ") + " …", "ends with": "… " + r["chunk_text"][-45:].replace("\n", " ")}
    for r in chunk_corpus.records if r["source_id"] == first_article
][:6]))

25,706 chunks from 2,572 articles · run chunk_by_size-bdd6bcf0, derived from ['ingest_corpus-b4fe4cee']
tokens per chunk: {'min': 41, 'median': 200.0, 'max': 200}


,id,tokens,starts with,ends with
0,97824#0,0–200,Enterprises are looking for increasingly powerful compute to …,"… hin the expanded 72-GPU NVIDIA NVLink domain,"
1,97824#1,160–360,"the show, Oracle also previewed NVIDIA GB200 NVL72 liquid-co …",… ee NVIDIA L4 Tensor Core GPUs. Companies are
2,97824#2,320–520,"provide scalable AI at the edge accelerated by NVIDIA GPUs, …",… how the NVIDIA accelerated computing platform
3,97824#3,480–680,GPU support for Oracle Machine Learning notebooks to allow c …,… AI to their structured and unstructured data
4,97824#4,640–840,and translation use cases across a range of model sizes and …,… equirements. Communication and collaboration
5,97824#5,800–1000,a leading global provider of consulting services and system …,"… se software platform, available on the Oracle"


The token ranges overlap by 40: neighbouring windows share their edges, so a statement cut at
one boundary still appears whole in the next window. And the windows start and end mid-sentence
— the baseline's weakness, and your opening for a better method.

**What counts as a token?** With the default `tokenizer="whitespace"`, a token is a word. An
embedding model counts differently. Pass its tokenizer — e.g.
`tokenizer="sentence-transformers/all-MiniLM-L6-v2"` — to size windows by what the model
actually reads, leaving a few tokens of margin: the model adds its own special tokens.

In [7]:
if HAS_EMBEDDINGS:
    from anatoolbox.preprocess.chunk.chunk_by_size import huggingface_spans

    minilm = huggingface_spans("sentence-transformers/all-MiniLM-L6-v2")
    counts = pd.Series([len(minilm(r["chunk_text"])) for r in chunk_corpus.records[:500]])
    print(f"a 200-word window is a median {counts.median():.0f} MiniLM tokens; "
          f"{(counts > 256).mean():.0%} exceed the 256 tokens the model reads")

a 200-word window is a median 250 MiniLM tokens; 38% exceed the 256 tokens the model reads


**Chunks lose context — a blueprint.** A chunk saying "the platform cut processing time by
30%" does not say which platform. `contextualize="title"` prepends the article's title to every
chunk before it is indexed: the simplest form of *contextual retrieval*. Richer context is yours to
add — write a function `(record, chunk_text) -> str`, register it, and select it by name. Whether
any of it helps retrieval on your questions is something to measure in Day 4, not assume.

In [8]:
from anatoolbox.preprocess.chunk.chunk_by_size import register_contextualizer

titled = chunk.run({"input": articles, "size": 200, "overlap": 40, "contextualize": "title"}, context=ctx)
print(get_corpus(titled["corpus"]).records[1]["text"][:220], "…\n")

# Your own contextualizer: any function (record, chunk_text) -> str, selected by name.
register_contextualizer("title_and_date", lambda record, chunk_text: f"{record['title']} ({record['date']})")
dated = chunk.run({"input": articles, "size": 200, "overlap": 40, "contextualize": "title_and_date"}, context=ctx)
print(get_corpus(dated["corpus"]).records[1]["text"][:220], "…")

NVIDIA and Oracle to Accelerate AI, Data Processing for Enterprises

the show, Oracle also previewed NVIDIA GB200 NVL72 liquid-cooled bare-metal instances to help power generative AI applications. The instances are capab …



NVIDIA and Oracle to Accelerate AI, Data Processing for Enterprises (2024-09-11)

the show, Oracle also previewed NVIDIA GB200 NVL72 liquid-cooled bare-metal instances to help power generative AI applications. The instan …


## 4 · Rewriting the question

A compound question retrieves evidence for only one of its parts; a broad one retrieves a
little of everything. `rewrite_query_for_retrieval` turns the question into retrieval
queries — it never answers it. `decompose` writes one query per part; `expand` writes
several phrasings. `exact_terms` are rare names worth matching literally, and `time_range`
captures a period *the question* names.

In [9]:
REWRITES = {}
if LLM_READY:
    for strategy in ("decompose", "expand"):
        started = time.perf_counter()
        REWRITES[strategy] = rewrites = rewrite.run(
            {"question": QUESTION, "strategy": strategy, "collection": "AI news articles, Sep 2024 – Aug 2025"},
            context=ctx,
        )
        print(f"\n{strategy} ({time.perf_counter() - started:.1f} s, {rewrites['model']})")
        display(pd.DataFrame(rewrites["queries"]))
        print("exact terms:", rewrites["exact_terms"], "| time range:", rewrites["time_range"], "| fallback:", rewrites["fallback"])
else:
    print("No language model configured — skipping query rewriting (see section 1).")
DECOMPOSED = REWRITES.get("decompose")


decompose (19.9 s, Qwen3-0.6B)


,query,purpose
0,What security risks do autonomous browser agents create?,Find security risks associated with autonomous browser agents
1,How are companies responding to these security risks?,Find responses companies are taking to security risks


exact terms: [] | time range: {'from': None, 'to': None} | fallback: False



expand (26.8 s, Qwen3-0.6B)


,query,purpose
0,What security risks do autonomous browser agents create?,Find security risks associated with autonomous browser agents
1,How are companies responding to security risks created by autonomous browser agents?,Find responses from companies to security risks
2,What security measures are companies implementing to address the risks created by autonomous browser agents?,Find measures taken by companies to mitigate security risks


exact terms: [] | time range: {'from': None, 'to': None} | fallback: False


## 5 · Retrieval with every rewrite, fused

Passing `queries_input` ranks the original question and each rewrite separately, then fuses
the rankings, so a passage relevant to *any* part of the question can surface. The overlap
shows how much the rewrites change what is retrieved.

Every call below passes the chunk result to search as `input` — the plain chunks, not the
header variant from section 3. Without recordset memory nothing is picked implicitly, so a
tool can never quietly search whichever corpus happens to be newest.

In [10]:
CHUNKS = chunks  # the plain chunks, not the title-context variants
single = retrieve.run({"query": QUESTION, "size": 30, "input": CHUNKS}, context=ctx)
fused = retrieve.run({"query": QUESTION, "size": 30, "input": CHUNKS, **({"queries_input": DECOMPOSED} if DECOMPOSED else {})}, context=ctx)
single_ids = {p["id"] for p in single["passages"]}
fused_ids = {p["id"] for p in fused["passages"]}
print(f"fused retrieval derived from: {fused['provenance']['derived_from']}")
print(f"top-30 overlap: {len(single_ids & fused_ids)} shared, {len(fused_ids - single_ids)} new passages from the rewrites")


def passage_table(result, n=8, score="score"):
    return pd.DataFrame([
        {"rank": p["rank"], score: round(p.get(score, 0), 3), "date": p.get("date"), "article": str(p.get("title"))[:70],
         "text": p["snippet"][:100].replace("\n", " ") + " …"}
        for p in result["passages"][:n]
    ])


display(passage_table(fused))

fused retrieval derived from: ['chunk_by_size-bdd6bcf0', 'rewrite_query_for_retrieval-144b55c1']
top-30 overlap: 20 shared, 10 new passages from the rewrites


,rank,score,date,article,text
0,1,0.041,2024-10-08,Accenture's Insight on AI-Driven Cybersecurity Future,"AI architecture, such as orchestration layers, RAG databases, or API integrations. Our understandin …"
1,2,0.040,2025-03-30,5 Ways to Protect Your Organization Against Evolving Digital Threats,"27001, and GDPR focus on data security and access controls, but they weren’ t built for AI agents. W …"
2,3,0.038,2025-07-23,Early Anthropic hire raises $ 15M to insure AI agents and help startup,"about unpredictable failures, liability issues, and reputational risks. AIUC’ s solution centers on …"
3,4,0.037,2024-10-29,Zenity Raises $ 38 Million Series B Funding Round to Secure Agentic AI,"s largest and most consequential organizations to enable innovation securely. ” Recently, Zenity re …"
4,5,0.033,2025-04-28,How Agentic AI Enables the Next Leap in Cybersecurity,Agentic AI is redefining the cybersecurity landscape — introducing new opportunities that demand ret …
5,6,0.032,2025-03-03,AI and AI-agents: A game-changer for cybersecurity and cybercrime,"to misleading outputs, discriminatory decision-making, or security misjudgments, potentially exacerb …"
6,7,0.032,2024-12-18,What is an AI agent? A computer scientist explains the next wave of ar,new domain before. Are AI agents poised to revolutionize the way humans work? This will depend on w …
7,8,0.031,2025-07-02,Bright Data beat Elon Musk and Meta in court — now its $ 100M AI platf,"engine designed to answer complex, multi-layered business questions in real-time. Unlike general-pur …"


**Time and metadata.** Trend questions care about *when*. `recency_half_life_days` makes a
passage that is one half-life older score half as much, measured from the newest date in the
corpus; `filters` restricts results by metadata such as the publishing site.

In [11]:
recent = retrieve.run({"query": QUESTION, "size": 5, "input": CHUNKS, "recency_half_life_days": 60}, context=ctx)
print("with a 60-day half-life, reference", recent["recency_reference_date"], "→ dates:", [p["date"] for p in recent["passages"]])
top_domains = track_df["domain"].value_counts().index[:2].tolist()
filtered = retrieve.run({"query": QUESTION, "size": 5, "input": CHUNKS, "filters": {"domain": top_domains}}, context=ctx)
print("filtered to", top_domains, "→", [p["domain"] for p in filtered["passages"]])

with a 60-day half-life, reference 2025-08-24 → dates: ['2025-08-24', '2025-08-10', '2025-08-11', '2025-08-04', '2025-08-11']


filtered to ['venturebeat', 'zbrain'] → ['venturebeat', 'venturebeat', 'venturebeat', 'venturebeat', 'venturebeat']


## 6 · Reranking the candidates

Retrieval is fast but coarse. `rerank_passages` reads each candidate's **full text** together
with the question and re-scores it with a cross-encoder — slow, which is why it runs on 30
candidates, not the corpus. `rank_change` shows what reranking did.

Chunking has a side effect here: several chunks of one article can fill most of the slots.
`max_per_source=2` keeps at most two passages per article.

In [12]:
reranked = None
if HAS_EMBEDDINGS:
    started = time.perf_counter()
    reranked = rerank.run({"keep": 6, "max_per_source": 2, "input": fused}, context=ctx)
    print(f"reranked {reranked['candidates']} candidates with {reranked['reranker']} in {time.perf_counter() - started:.1f} s"
          f" ({reranked['skipped_over_source_limit']} skipped by the per-article limit)")
    display(pd.DataFrame([
        {"rank": p["rank"], "was": p["retrieval_rank"], "change": p["rank_change"], "date": p.get("date"), "article": str(p.get("title"))[:80]}
        for p in reranked["passages"]
    ]))
    ANSWER_INPUT = reranked
else:
    print("sentence-transformers is not installed — answering from the top retrieved passages instead.")
    ANSWER_INPUT = fused

reranked 30 candidates with cross-encoder/ms-marco-MiniLM-L-6-v2 in 4.8 s (0 skipped by the per-article limit)


,rank,was,change,date,article
0,1,30,29,2025-07-12,Securing the Next Frontier: Why Agentic AI Demands a New Approach to Cybersecuri
1,2,10,8,2024-10-25,The AI Agents Are Coming — Is Your Brand Ready?
2,3,5,2,2025-04-28,How Agentic AI Enables the Next Leap in Cybersecurity
3,4,2,-2,2025-03-30,5 Ways to Protect Your Organization Against Evolving Digital Threats
4,5,9,4,2025-01-23,"AI agents spark interest, concern for businesses in 2025"
5,6,25,19,2025-02-05,"AI Agents: Current Status, Industry Impact, and Job Market Implications"


## 7 · Writing a cited answer

`synthesize_answer` labels the passages `[S1]…[Sn]`, drops duplicates, and places the most
relevant passages at the start and end of the prompt (models attend least to the middle).
Two modes:

- **grounded** — only what the sources say; say so when they are not enough.
- **blended** — may add background knowledge, marked as background and never cited.

The tool does not trust the model's citations. It reports **unknown citations** (labels that
were never provided), **uncited sources**, and **citation coverage** — the share of sentences
that carry an inline citation. None of these proves an answer faithful; Day 4 adds that
evaluation. They are cheap, objective signals, and small models trip them often.

In [13]:
answers = {}
if LLM_READY:
    for mode in ("grounded", "blended"):
        started = time.perf_counter()
        answers[mode] = synthesize.run(
            {"question": QUESTION, "mode": mode, "input": ANSWER_INPUT, "max_per_source": 2, "max_chars_per_passage": 1200, "max_tokens": 400},
            context=ctx,
        )
        answer = answers[mode]
        display(Markdown(f"### {mode} · {answer['model']} · {time.perf_counter() - started:.0f} s\n\n{answer['answer_markdown']}"))
else:
    print("No language model configured — skipping answer synthesis (see section 1).")

### grounded · Qwen3-0.6B · 38 s

Autonomous browser agents introduce security risks due to their ability to make independent decisions and interact with systems, potentially leading to vulnerabilities. Companies are responding by implementing robust security measures, such as data encryption, opt-in protocols, and retention policies, to address these risks. For example, [S3](https://blogs.nvidia.com/blog/agentic-ai-cybersecurity/) mentions that enterprises must adopt a dual approach to secure AI while mitigating risks. Additionally, [S4](https://aijourn.com/5-ways-to-protect-your-organization-against-evolving-digital-threats/) highlights the need for comprehensive testing protocols to ensure AI-generated code remains secure.

### blended · Qwen3-0.6B · 64 s

Autonomous browser agents introduce significant security risks due to their ability to make independent decisions and interact with systems without human oversight. These agents can process sensitive data, execute tasks autonomously, and potentially bypass traditional security measures. Companies are responding by implementing robust security frameworks, enhancing data protection, and ensuring transparency and accountability in AI operations.

For example, [S3](https://blogs.nvidia.com/blog/agentic-ai-cybersecurity/) highlights that enterprises must adopt a dual approach to cybersecurity, combining defense and attack prevention. [S5](https://www.ciodive.com/news/enterprise-AI-agent-agentic-autonomous-strategy-challenges/738172/) notes that AI agent abuse is a growing concern, with Gartner predicting it will be a leading cause of enterprise breaches. [S4](https://aijourn.com/5-ways-to-protect-your-organization-against-evolving-digital-threats/) emphasizes the need for comprehensive policies to address AI-specific security challenges, including model bias and data lineage tracking. [S6](https://bytebridge.medium.com/ai-agents-current-status-industry-impact-and-job-market-implications-f8f1ccd0e01f) discusses the potential risks of AI agents, such as data leaks and unauthorized changes, and mentions measures like data encryption and opt-in protocols.

In [14]:
if answers:
    display(pd.DataFrame([
        {"mode": mode, "cited": ", ".join(a["cited"]) or "—", "unknown citations": ", ".join(a["unknown_citations"]) or "—",
         "uncited sources": ", ".join(a["uncited_sources"]) or "—",
         "sentences with a citation": f"{a['statements_with_citations']} of {a['statements']}", "coverage": a["citation_coverage"]}
        for mode, a in answers.items()
    ]))
    any_answer = next(iter(answers.values()))
    display(pd.DataFrame(any_answer["sources"])[["label", "rank", "position", "date", "title", "url"]])

,mode,cited,unknown citations,uncited sources,sentences with a citation,coverage
0,grounded,"S3, S4",—,"S1, S2, S5, S6",2 of 4,0.500
1,blended,"S3, S5, S4, S6",—,"S1, S2",4 of 7,0.571


,label,rank,position,date,title,url
0,S1,1,1,2025-07-12,Securing the Next Frontier: Why Agentic AI Demands a New Approach to Cybersecurity,https://aijourn.com/securing-the-next-frontier-why-agentic-ai-demands-a-new-approach-to-cybersecurity/
1,S2,2,6,2024-10-25,The AI Agents Are Coming — Is Your Brand Ready?,https://medium.com/ipg-media-lab/the-ai-agents-are-coming-is-your-brand-ready-f6c493b1ccb0
2,S3,3,2,2025-04-28,How Agentic AI Enables the Next Leap in Cybersecurity,https://blogs.nvidia.com/blog/agentic-ai-cybersecurity/
3,S4,4,5,2025-03-30,5 Ways to Protect Your Organization Against Evolving Digital Threats,https://aijourn.com/5-ways-to-protect-your-organization-against-evolving-digital-threats/
4,S5,5,3,2025-01-23,"AI agents spark interest, concern for businesses in 2025",https://www.ciodive.com/news/enterprise-AI-agent-agentic-autonomous-strategy-challenges/738172/
5,S6,6,4,2025-02-05,"AI Agents: Current Status, Industry Impact, and Job Market Implications",https://bytebridge.medium.com/ai-agents-current-status-industry-impact-and-job-market-implications-f8f1ccd...


**Reading these results.** With a small model, expect trouble — and expect it to vary
between runs. In runs of this notebook, Qwen3-0.6B has written grounded answers with no
citation at all, collected citations at the end of a paragraph instead of after each
sentence, written a source list it was told not to write, and made statements the checks
cannot confirm the sources support. None of that is a reason to hide the model's output —
it is exactly what your evaluation has to catch, and why model choice is a Stage 3 design
decision.

## 8 · Lineage: the whole loop on one page

Every result carries its provenance: the settings it ran with and the `run_id`s it was derived
from. When an evaluation in Day 4 flags an answer, this is how you find out whether the
chunking, the rewrite, the retrieval, the reranking, or the model was responsible. It is plain
JSON, so the cell below also saves it next to the data.

In [15]:
import json

from anatoolbox.provenance import lineage

steps = {"ingest": articles, "chunk": chunks, "chunk + title": titled, "chunk + title & date": dated}
steps.update({f"rewrite ({strategy})": result for strategy, result in REWRITES.items()})
steps.update({"retrieve": single, "retrieve (fused)": fused, "retrieve (recency)": recent, "retrieve (filtered)": filtered})
if reranked:
    steps["rerank"] = reranked
steps.update({f"answer ({mode})": result for mode, result in answers.items()})

SHOWN = ("strategy", "size", "overlap", "contextualize", "queries", "keep", "max_per_source",
         "mode", "model", "reranker", "recency_half_life_days", "filters")
table = pd.DataFrame(lineage(steps.values()))
table.insert(0, "step", list(steps))
table["derived_from"] = table["derived_from"].map(lambda refs: ", ".join(refs) or "—")
table["settings"] = table["settings"].map(lambda s: {k: v for k, v in s.items() if k in SHOWN and v not in (None, [], {})})
display(table)

provenance_file = DATA_DIR / "day3_provenance.json"
provenance_file.write_text(json.dumps({step: result["provenance"] for step, result in steps.items()}, indent=2))
print(f"saved {len(steps)} provenance records to {provenance_file.name}")
if answers:
    print("\none record in full:")
    print(json.dumps(next(iter(answers.values()))["provenance"], indent=2))

,step,run_id,tool,derived_from,settings
0,ingest,ingest_corpus-b4fe4cee,ingest_corpus,—,{}
1,chunk,chunk_by_size-bdd6bcf0,chunk_by_size,ingest_corpus-b4fe4cee,"{'size': 200, 'overlap': 40, 'contextualize': 'none'}"
2,chunk + title,chunk_by_size-b22d43f2,chunk_by_size,ingest_corpus-b4fe4cee,"{'size': 200, 'overlap': 40, 'contextualize': 'title'}"
3,chunk + title & date,chunk_by_size-5649940e,chunk_by_size,ingest_corpus-b4fe4cee,"{'size': 200, 'overlap': 40, 'contextualize': 'title_and_date'}"
4,rewrite (decompose),rewrite_query_for_retrieval-144b55c1,rewrite_query_for_retrieval,—,"{'strategy': 'decompose', 'model': 'Qwen3-0.6B'}"
5,rewrite (expand),rewrite_query_for_retrieval-d922e8f0,rewrite_query_for_retrieval,—,"{'strategy': 'expand', 'model': 'Qwen3-0.6B'}"
6,retrieve,retrieve_passages-0bb51c31,retrieve_passages,chunk_by_size-bdd6bcf0,"{'strategy': 'sparse', 'size': 30}"
7,retrieve (fused),retrieve_passages-510c1a0c,retrieve_passages,"chunk_by_size-bdd6bcf0, rewrite_query_for_retrieval-144b55c1","{'strategy': 'sparse', 'size': 30, 'queries': ['What security risks do autonomous browser agents create, a..."
8,retrieve (recency),retrieve_passages-391aad8a,retrieve_passages,chunk_by_size-bdd6bcf0,"{'strategy': 'sparse', 'size': 5, 'recency_half_life_days': 60.0}"
9,retrieve (filtered),retrieve_passages-66229dd6,retrieve_passages,chunk_by_size-bdd6bcf0,"{'strategy': 'sparse', 'size': 5, 'filters': {'domain': ['venturebeat', 'zbrain']}}"


saved 13 provenance records to day3_provenance.json

one record in full:
{
  "run_id": "synthesize_answer-5a0cb257",
  "tool": "synthesize_answer",
  "anatoolbox_version": "0.1.0",
  "created_at": "2026-09-15T06:48:17+00:00",
  "settings": {
    "question": "What security risks do autonomous browser agents create, and how are companies responding?",
    "mode": "grounded",
    "order": "edges",
    "model": "Qwen3-0.6B",
    "max_passages": 8,
    "max_chars_per_passage": 1200,
    "max_per_source": 2,
    "max_tokens": 400,
    "instructions": null
  },
  "derived_from": [
    "rerank_passages-012d3d98"
  ]
}


## What to take away

- **Every Stage 3 enhancement the brief lists is now an argument:** chunk size, query
  rewriting, reranking, retrieval filtering. Comparing variants means changing a setting and
  keeping the lineage, not rebuilding the pipeline.
- **Model choice is visible and swappable.** Any OpenAI-compatible endpoint works, the model
  used is recorded with every answer, and roles let rewriting and answering use different models.
- **Notebooks and pipelines need no memory layer.** Results chain directly, and every result
  carries portable provenance — saved as JSON, it tells you later exactly which settings and
  inputs produced each answer.
- **Citations are checked, not trusted.** Unknown labels, uncited sources and citation coverage
  are objective signals — and a small local model shows why you need them.

**Next — Day 4: evaluation.** Retrieval metrics (precision, recall, MRR), answer faithfulness
and relevance with an LLM as judge, and a Q&A dataset built with a *different* model — plus
the hand-off that brings your Stage 1 knowledge graph into retrieval.